# NetworkPulse AI — One-Day Multi-Model Research

This notebook evaluates multiple machine-learning approaches on one CICIDS2017 day-level flow dataset.

Models evaluated:
- Random Forest
- XGBoost
- LightGBM
- CatBoost
- Isolation Forest
- One-Class SVM
- Local Outlier Factor
- DBSCAN
- Autoencoder
- XGBoost + LightGBM + CatBoost Ensemble

The notebook produces quantitative evaluation, feature importance, attack distribution, model comparison, and an evidence-based conclusion for the selected day.


In [ ]:
!pip -q install xgboost lightgbm catboost openpyxl

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## 1. Upload the day-level CICIDS2017 CSV

Use one of the supplied day files such as:
- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
- Friday-WorkingHours-Morning.pcap_ISCX.csv
- Monday-WorkingHours.pcap_ISCX.csv
- Tuesday-WorkingHours.pcap_ISCX.csv
- Wednesday-workingHours.pcap_ISCX.csv
- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
- Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv


In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded.keys()))
print(file_name)


In [ ]:
df = pd.read_csv(file_name, low_memory=False)
print("Shape:", df.shape)
display(df.head())
display(df[" Label"].value_counts(dropna=False) if " Label" in df.columns else df["Label"].value_counts(dropna=False))


## 2. Data preparation

The target is converted into binary classification:
- BENIGN → 0
- any attack label → 1


In [ ]:
df.columns = [str(c).strip() for c in df.columns]

label_col = "Label"

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(axis=0, how="all")

y_raw = df[label_col].astype(str).str.strip()
y = (y_raw != "BENIGN").astype(int)

attack_distribution = y_raw.value_counts().rename_axis("Attack_Type").reset_index(name="Count")
display(attack_distribution)

X = df.drop(columns=[label_col]).copy()

for c in X.columns:
    if X[c].dtype == "object":
        X[c] = pd.to_numeric(X[c], errors="coerce")

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols)

X = X.astype(np.float32)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("Benign:", int((y == 0).sum()))
print("Attack:", int((y == 1).sum()))


In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=attack_distribution.head(20), x="Attack_Type", y="Count")
plt.xticks(rotation=75, ha="right")
plt.title("Attack Distribution")
plt.tight_layout()
plt.show()


### Inference

The selected day contains a mixture of benign and attack traffic. The class balance is important because accuracy alone can become misleading when attack prevalence is low or highly concentrated in one category.


## 3. Train-test split

A stratified hold-out split is used so that the benchmark contains both benign and attack observations in training and testing.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


## 4. Supervised models

The supervised research layer compares:
- Random Forest
- XGBoost
- LightGBM
- CatBoost


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.08,
        num_leaves=63,
        max_depth=-1,
        random_state=42,
        verbosity=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=8,
        learning_rate=0.08,
        loss_function="Logloss",
        verbose=False,
        random_seed=42
    )
}

results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    predictions[name] = pred
    probabilities[name] = prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

supervised_results = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
display(supervised_results)


### Inference

The comparison identifies which supervised learner provides the strongest balance between attack recall and false-positive control on this specific day. F1 and ROC-AUC are emphasized because intrusion detection is not adequately represented by accuracy alone.


In [ ]:
best_supervised = supervised_results.iloc[0]["Model"]
print("Best supervised model by F1:", best_supervised)


## 5. Ensemble detector

The three principal gradient-boosting learners are combined using probability averaging:
XGBoost + LightGBM + CatBoost.


In [ ]:
ensemble_prob = (
    probabilities["XGBoost"] +
    probabilities["LightGBM"] +
    probabilities["CatBoost"]
) / 3.0

ensemble_pred = (ensemble_prob >= 0.5).astype(int)

ensemble_metrics = {
    "Model": "XGBoost + LightGBM + CatBoost Ensemble",
    "Accuracy": accuracy_score(y_test, ensemble_pred),
    "Precision": precision_score(y_test, ensemble_pred, zero_division=0),
    "Recall": recall_score(y_test, ensemble_pred, zero_division=0),
    "F1": f1_score(y_test, ensemble_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, ensemble_prob)
}

results_with_ensemble = pd.concat(
    [supervised_results, pd.DataFrame([ensemble_metrics])],
    ignore_index=True
).sort_values("F1", ascending=False).reset_index(drop=True)

display(results_with_ensemble)


### Inference

Probability-level ensemble averaging reduces dependence on a single learner and provides the model-agreement evidence used by NetworkPulse AI when explaining high-risk events.


## 6. Isolation Forest

Isolation Forest provides the unsupervised anomaly-detection baseline.


In [ ]:
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train_scaled)

iso_score = -iso.decision_function(X_test_scaled)
iso_pred = (iso.predict(X_test_scaled) == -1).astype(int)

threshold = np.percentile(iso_score, 97)
iso_pred_threshold = (iso_score >= threshold).astype(int)

iso_results = {
    "Model": "Isolation Forest",
    "Accuracy": accuracy_score(y_test, iso_pred_threshold),
    "Precision": precision_score(y_test, iso_pred_threshold, zero_division=0),
    "Recall": recall_score(y_test, iso_pred_threshold, zero_division=0),
    "F1": f1_score(y_test, iso_pred_threshold, zero_division=0)
}

display(pd.DataFrame([iso_results]))


## 7. One-Class SVM


In [ ]:
normal_train = X_train_scaled[y_train.to_numpy() == 0]

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

ocsvm.fit(normal_train)

ocsvm_pred = (ocsvm.predict(X_test_scaled) == -1).astype(int)

ocsvm_results = {
    "Model": "One-Class SVM",
    "Accuracy": accuracy_score(y_test, ocsvm_pred),
    "Precision": precision_score(y_test, ocsvm_pred, zero_division=0),
    "Recall": recall_score(y_test, ocsvm_pred, zero_division=0),
    "F1": f1_score(y_test, ocsvm_pred, zero_division=0)
}

display(pd.DataFrame([ocsvm_results]))


## 8. Local Outlier Factor


In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=35,
    contamination=0.05,
    novelty=True,
    n_jobs=-1
)

lof.fit(normal_train)

lof_pred = (lof.predict(X_test_scaled) == -1).astype(int)

lof_results = {
    "Model": "Local Outlier Factor",
    "Accuracy": accuracy_score(y_test, lof_pred),
    "Precision": precision_score(y_test, lof_pred, zero_division=0),
    "Recall": recall_score(y_test, lof_pred, zero_division=0),
    "F1": f1_score(y_test, lof_pred, zero_division=0)
}

display(pd.DataFrame([lof_results]))


## 9. DBSCAN


In [ ]:
sample_size = min(20000, len(X_test_scaled))
sample_idx = np.random.RandomState(42).choice(
    len(X_test_scaled),
    size=sample_size,
    replace=False
)

db = DBSCAN(
    eps=1.5,
    min_samples=10,
    n_jobs=-1
)

db_labels = db.fit_predict(X_test_scaled[sample_idx])
db_pred = (db_labels == -1).astype(int)

db_y = y_test.iloc[sample_idx]

db_results = {
    "Model": "DBSCAN",
    "Accuracy": accuracy_score(db_y, db_pred),
    "Precision": precision_score(db_y, db_pred, zero_division=0),
    "Recall": recall_score(db_y, db_pred, zero_division=0),
    "F1": f1_score(db_y, db_pred, zero_division=0)
}

display(pd.DataFrame([db_results]))


## 10. Autoencoder

The Autoencoder is trained primarily on normal traffic. Reconstruction error is then used as an anomaly score.


In [ ]:
normal_scaled = X_train_scaled[y_train.to_numpy() == 0]

input_dim = normal_scaled.shape[1]

inp = Input(shape=(input_dim,))
enc = Dense(64, activation="relu")(inp)
enc = Dense(32, activation="relu")(enc)
latent = Dense(16, activation="relu")(enc)
dec = Dense(32, activation="relu")(latent)
dec = Dense(64, activation="relu")(dec)
out = Dense(input_dim, activation="linear")(dec)

autoencoder = Model(inp, out)
autoencoder.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

autoencoder.fit(
    normal_scaled,
    normal_scaled,
    epochs=20,
    batch_size=256,
    validation_split=0.10,
    shuffle=True,
    verbose=0
)

normal_reconstruction = autoencoder.predict(normal_scaled, verbose=0)
normal_error = np.mean(
    np.square(normal_scaled - normal_reconstruction),
    axis=1
)

ae_threshold = np.percentile(normal_error, 97)

test_reconstruction = autoencoder.predict(X_test_scaled, verbose=0)
test_error = np.mean(
    np.square(X_test_scaled - test_reconstruction),
    axis=1
)

ae_pred = (test_error >= ae_threshold).astype(int)

ae_results = {
    "Model": "Autoencoder",
    "Accuracy": accuracy_score(y_test, ae_pred),
    "Precision": precision_score(y_test, ae_pred, zero_division=0),
    "Recall": recall_score(y_test, ae_pred, zero_division=0),
    "F1": f1_score(y_test, ae_pred, zero_division=0)
}

display(pd.DataFrame([ae_results]))
print("Autoencoder threshold:", ae_threshold)


### Inference

The Autoencoder provides a fundamentally different signal from supervised classifiers: it models normal traffic and identifies flows whose reconstruction error deviates substantially from normal behaviour.


## 11. Combined research comparison


In [ ]:
unsupervised_results = pd.DataFrame([
    iso_results,
    ocsvm_results,
    lof_results,
    db_results,
    ae_results
])

comparison = pd.concat(
    [results_with_ensemble, unsupervised_results],
    ignore_index=True,
    sort=False
)

comparison = comparison[
    ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
]

comparison = comparison.sort_values(
    "F1",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

display(comparison)


In [ ]:
plt.figure(figsize=(13,7))
plot_df = comparison.copy()
plot_df["F1_percent"] = plot_df["F1"] * 100
sns.barplot(data=plot_df, x="F1_percent", y="Model")
plt.xlabel("F1 Score (%)")
plt.ylabel("")
plt.title("Model Comparison by F1 Score")
plt.tight_layout()
plt.show()


### Research inference

The strongest detector is selected according to the evidence generated on the selected day rather than by assuming that one algorithm is universally superior. The ensemble is especially useful when its component learners show high agreement.


## 12. Feature importance

XGBoost, LightGBM and CatBoost importance rankings are compared to identify recurring traffic characteristics associated with the detection decision.


In [ ]:
importance_frames = []

for name in ["XGBoost", "LightGBM", "CatBoost"]:
    model = models[name]
    values = model.feature_importances_

    temp = pd.DataFrame({
        "Feature": X.columns,
        "Importance": values,
        "Model": name
    }).sort_values("Importance", ascending=False)

    importance_frames.append(temp)

importance = pd.concat(importance_frames, ignore_index=True)

top_importance = (
    importance.groupby("Feature")["Importance"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

display(top_importance)


In [ ]:
plt.figure(figsize=(12,7))
sns.barplot(data=top_importance, x="Importance", y="Feature")
plt.title("Average Feature Importance Across XGBoost, LightGBM and CatBoost")
plt.tight_layout()
plt.show()


## 13. Attack-category analysis

This separates the overall binary detection result from the behaviour of individual attack families.


In [ ]:
test_indices = X_test.index

test_attack_types = y_raw.loc[test_indices].reset_index(drop=True)
test_results = pd.DataFrame({
    "Attack_Type": test_attack_types,
    "Actual": y_test.reset_index(drop=True),
    "Ensemble_Probability": ensemble_prob,
    "Ensemble_Prediction": ensemble_pred,
    "Autoencoder_Prediction": ae_pred
})

category_summary = (
    test_results
    .groupby("Attack_Type")
    .agg(
        Samples=("Attack_Type", "size"),
        Attack_Rate=("Actual", "mean"),
        Ensemble_Mean_Probability=("Ensemble_Probability", "mean"),
        Predicted_Attack_Rate=("Ensemble_Prediction", "mean")
    )
    .sort_values("Samples", ascending=False)
)

display(category_summary)


In [ ]:
attack_only = test_results[test_results["Actual"] == 1]

category_recall = (
    attack_only.groupby("Attack_Type")
    .apply(lambda g: (g["Ensemble_Prediction"] == 1).mean())
    .rename("Recall")
    .sort_values(ascending=False)
    .reset_index()
)

display(category_recall)

plt.figure(figsize=(12,7))
plot_cat = category_recall.copy()
plot_cat["Recall_percent"] = plot_cat["Recall"] * 100
sns.barplot(data=plot_cat, x="Recall_percent", y="Attack_Type")
plt.xlabel("Recall (%)")
plt.ylabel("")
plt.title("Ensemble Recall by Attack Category")
plt.tight_layout()
plt.show()


### Inference

Category-level variation is critical. A strong aggregate F1 can hide weak detection of specific attack families, so the category recall table is retained as part of the final evidence.


## 14. Model agreement on high-risk flows


In [ ]:
agreement = pd.DataFrame({
    "XGBoost": probabilities["XGBoost"],
    "LightGBM": probabilities["LightGBM"],
    "CatBoost": probabilities["CatBoost"],
    "Ensemble": ensemble_prob
})

agreement["Max_Min_Gap"] = agreement[["XGBoost","LightGBM","CatBoost"]].max(axis=1) - agreement[["XGBoost","LightGBM","CatBoost"]].min(axis=1)

agreement["Agreement"] = 1 - agreement["Max_Min_Gap"]

top_risk = pd.DataFrame({
    "Row_Index": X_test.index,
    "Actual": y_test.to_numpy(),
    "XGBoost": probabilities["XGBoost"],
    "LightGBM": probabilities["LightGBM"],
    "CatBoost": probabilities["CatBoost"],
    "Ensemble": ensemble_prob,
    "Agreement": agreement["Agreement"].to_numpy()
})

top_risk = top_risk.sort_values(
    ["Ensemble","Agreement"],
    ascending=[False,False]
).head(20)

display(top_risk)


## 15. Confusion matrix for the selected champion


In [ ]:
champion_row = results_with_ensemble.iloc[0]["Model"]

if champion_row == "XGBoost + LightGBM + CatBoost Ensemble":
    champion_pred = ensemble_pred
elif champion_row in predictions:
    champion_pred = predictions[champion_row]
else:
    champion_pred = ensemble_pred

cm = confusion_matrix(y_test, champion_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="magma",
    xticklabels=["Benign","Attack"],
    yticklabels=["Benign","Attack"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {champion_row}")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    champion_pred,
    target_names=["Benign","Attack"],
    zero_division=0
))


## 16. One-day evidence conclusion


In [ ]:
best_row = comparison.iloc[0]

day_name = file_name.replace(".csv","")

conclusion = f'''
One-day research conclusion for {day_name}

The selected CICIDS2017 day was evaluated using supervised, unsupervised, clustering and deep-learning approaches.

The strongest measured model by F1 score was {best_row["Model"]}, achieving an F1 score of {best_row["F1"]:.4f}, precision of {best_row["Precision"]:.4f}, recall of {best_row["Recall"]:.4f}, and accuracy of {best_row["Accuracy"]:.4f}.

The XGBoost, LightGBM and CatBoost models were also evaluated individually and combined through probability averaging to form the NetworkPulse ensemble. Agreement among these models provides an additional confidence signal for high-risk flows.

Isolation Forest, One-Class SVM, Local Outlier Factor, DBSCAN and an Autoencoder were evaluated as complementary anomaly-detection approaches. These models provide a different perspective from supervised classification because they identify deviations from learned normal structure.

The category-level analysis shows whether detection quality is consistent across attack families. This prevents the final conclusion from relying only on aggregate binary metrics.

The results support the NetworkPulse design principle that an anomaly score should be treated as an investigation signal rather than as the final security decision. Model evidence, attack category, model agreement and contextual enrichment can then be used by the agent to explain why a flow was flagged and recommend the next investigative step.

These results describe the selected day and should not be interpreted as guaranteed performance on unseen networks or future traffic distributions.
'''

print(conclusion)


## 17. Export research evidence


In [ ]:
output_dir = Path("networkpulse_one_day_results")
output_dir.mkdir(exist_ok=True)

comparison.to_csv(output_dir / "model_comparison.csv", index=False)
attack_distribution.to_csv(output_dir / "attack_distribution.csv", index=False)
category_recall.to_csv(output_dir / "category_recall.csv", index=False)
top_risk.to_csv(output_dir / "top_risk_flows.csv", index=False)
top_importance.to_csv(output_dir / "feature_importance.csv", index=False)

with open(output_dir / "one_day_conclusion.txt", "w", encoding="utf-8") as f:
    f.write(conclusion)

import shutil
shutil.make_archive("NetworkPulse_AI_One_Day_Results", "zip", output_dir)

print("Created:")
print("NetworkPulse_AI_One_Day_Results.zip")


# Final viva interpretation

NetworkPulse AI uses multiple ML perspectives rather than depending on a single algorithm.

The supervised layer compares Random Forest, XGBoost, LightGBM and CatBoost.

The anomaly layer compares Isolation Forest, One-Class SVM, Local Outlier Factor, DBSCAN and Autoencoder.

The production-oriented ensemble combines XGBoost, LightGBM and CatBoost probabilities and uses model agreement as additional evidence.

For a selected day, the final conclusion is based on measured results from that day's flows. The notebook therefore provides a reproducible evidence trail from raw network traffic → preprocessing → model evaluation → attack-family analysis → high-risk flows → conclusion.
